# Aff-Wild2 Stage 1 — setup & pre-flight

Verifies that the ABAW-7 MTL-challenge frames release in `../competition-data/` is in the shape this repo expects, checks that the annotation parser consumes both splits without error, loads the two stage-1 configs, and smoke-tests the EmotiEffLib backbones so the heavy extraction in `aw2_01_extract_visual.ipynb` doesn't fail halfway through.

Run this once; it is read-only except for printing diagnostics.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve().parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

COMP_DATA = (REPO.parent / 'competition-data').resolve()
CROPPED_ALIGNED = COMP_DATA / 'cropped_aligned'
TRAIN_ANN = COMP_DATA / 'training_set_annotations.txt'
VAL_ANN   = COMP_DATA / 'validation_set_annotations.txt'

print('repo       :', REPO)
print('comp-data  :', COMP_DATA)
for p in [CROPPED_ALIGNED, TRAIN_ANN, VAL_ANN]:
    assert p.exists(), f'missing: {p}'
    print('  ok ->', p)

## 1. Inventory: folders and annotation counts

In [ ]:
video_dirs = sorted(p for p in CROPPED_ALIGNED.iterdir() if p.is_dir())
print('video folders in cropped_aligned :', len(video_dirs))
print('first 5 :', [p.name for p in video_dirs[:5]])

def count_lines(p):
    with open(p, 'r', encoding='utf-8') as fh:
        return sum(1 for _ in fh)

tr, va = count_lines(TRAIN_ANN), count_lines(VAL_ANN)
print(f'train annotations (incl. header): {tr:,}  -> {tr - 1:,} rows')
print(f'val   annotations (incl. header): {va:,}  -> {va - 1:,} rows')

assert len(video_dirs) == 307, f'expected 307 video folders, got {len(video_dirs)}'
assert tr - 1 == 142382
assert va - 1 == 26876
print('inventory matches plan.')

## 2. Annotation-parser smoke test

Runs `read_mtl_annotations` on both splits. `num_missed` stays at 0 while `features_index=None`; it only drops rows whose features are missing from a cache.

In [ ]:
from src.datasets.affwild2_mtl import read_mtl_annotations

for tag, p in [('train', TRAIN_ANN), ('val', VAL_ANN)]:
    a = read_mtl_annotations(p, features_index=None)
    va_valid   = int(a.mask_va.sum())
    expr_valid = int(a.mask_expr.sum())
    au_valid   = int(a.mask_au.sum())
    print(f'{tag:>5}: kept={len(a):>7,}  VA={va_valid:>7,}  EXPR={expr_valid:>7,}  AU={au_valid:>7,}  missed={a.num_missed}')

## 3. Sample image shape check

ABAW ReadMe: `cropped_aligned` is 112x112x3 JPEG. Assert on four random frames.

In [ ]:
from PIL import Image
import random

def sample_frame(video_dir):
    jpgs = [f for f in os.listdir(video_dir) if f.lower().endswith('.jpg')]
    return random.choice(jpgs)

rng = random.Random(0)
for vd in rng.sample(video_dirs, k=4):
    fn = sample_frame(vd)
    img = Image.open(vd / fn)
    print(f'{vd.name}/{fn}  size={img.size}  mode={img.mode}')
    assert img.size == (112, 112), f'unexpected size {img.size} for {vd.name}/{fn}'

## 4. Config parse check

Both stage-1 configs should load cleanly and point at paths that exist.

In [ ]:
from omegaconf import OmegaConf

for cfg_name in ['configs/aw2_stage1_enet.yaml', 'configs/aw2_stage1_mbf.yaml']:
    cfg = OmegaConf.load(cfg_name)
    print('---', cfg_name)
    print('run_name        :', cfg.run_name)
    print('backbone        :', cfg.backbone.model_name)
    print('features_cache  :', cfg.features_cache)
    print('results_dir     :', cfg.output.results_dir)
    for key in ['cropped_aligned_dir', 'train_annotations', 'val_annotations']:
        p = Path(cfg.data[key])
        assert p.exists(), f'config path {key}={p} does not resolve'
        print(f'  ok  {key} -> {p}')

## 5. Backbone smoke test

Instantiating `EmotiEffLibRecognizer` downloads weights on first use; flush once here so the extraction notebook starts from a warm cache. This cell needs `emotiefflib` installed (`pip install -e ../EmotiEffLib-main` or via `pyproject.toml`).

In [ ]:
import torch
import numpy as np
from emotiefflib.facial_analysis import EmotiEffLibRecognizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

for model_name in ['enet_b0_8_va_mtl', 'mbf_va_mtl']:
    rec = EmotiEffLibRecognizer(engine='torch', model_name=model_name, device=device)
    dummy = [np.zeros((112, 112, 3), dtype=np.uint8)]
    feats = rec.extract_features(dummy)
    _, scores = rec.classify_emotions(feats, logits=True)
    print(f'{model_name}: features.shape={np.asarray(feats).shape}  scores.shape={np.asarray(scores).shape}')
    del rec
    if device == 'cuda':
        torch.cuda.empty_cache()

If every cell above printed without raising, notebook `aw2_01_extract_visual.ipynb` is safe to run.